<a href="https://colab.research.google.com/github/JordanRD/PT01_GenerativeAI/blob/main/Project_Team_Group_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch datasets accelerate

import torch
from datasets import load_dataset
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from transformers import DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments

model_name = "gpt2"

print("Memuat Tokenizer dan Model bawaan...")
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

print("Mengunduh dataset cerita anak (TinyStories)...")
dataset = load_dataset("roneneldan/TinyStories", split="train[:1%]")


def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

print("Memproses tokenisasi pada dataset...")

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False
)

training_args = TrainingArguments(
    output_dir="./gpt2-storybook",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets,
)

print("\nMemulai proses Fine-Tuning...")
trainer.train()
print("Fine-Tuning Selesai!\n")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def generate_story(age, genre, theme, strategy="top_p"):
    prompt = f"Write a {genre} story for a {age} year old about {theme}."
    print(f"[Prompt Input]: {prompt}")

    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    gen_kwargs = {
        "input_ids": input_ids,
        "max_length": 150,
        "no_repeat_ngram_size": 2,
        "pad_token_id": tokenizer.eos_token_id,
        "do_sample": True,
    }

    if strategy == "top_k":
        gen_kwargs["top_k"] = 40
        gen_kwargs["temperature"] = 0.7
        print(f"--- Hasil Generasi (Top-K Sampling) ---")

    elif strategy == "top_p":
        gen_kwargs["top_p"] = 0.9
        gen_kwargs["temperature"] = 0.8
        print(f"--- Hasil Generasi (Top-p / Nucleus Sampling) ---")

    output_ids = model.generate(**gen_kwargs)

    story = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return story

umur = 7
genre_cerita = "fantasy"
tema_cerita = "a little dragon who wants to be a hero"

cerita_top_k = generate_story(umur, genre_cerita, tema_cerita, strategy="top_k")
print(cerita_top_k)

print("\n" + "="*60 + "\n")

cerita_top_p = generate_story(umur, genre_cerita, tema_cerita, strategy="top_p")
print(cerita_top_p)

Memuat Tokenizer dan Model bawaan...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Mengunduh dataset cerita anak (TinyStories)...
Memproses tokenisasi pada dataset...

Memulai proses Fine-Tuning...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,2.195731
200,2.039672
300,1.983873
400,1.945956
500,1.935139
600,1.905641
700,1.899384
800,1.863139
900,1.862940
1000,1.869589


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-Tuning Selesai!

[Prompt Input]: Write a fantasy story for a 7 year old about a little dragon who wants to be a hero.
--- Hasil Generasi (Top-K Sampling) ---
Write a fantasy story for a 7 year old about a little dragon who wants to be a hero. He has a big castle and lots of people. The dragon has two friends, a nice dragon and a friendly dragon.

The dragon wants a toy with wings. One day, he sees a new toy that looks like a dragon toy. It is red and shiny and has wings and scales. A dragon is fun and brave. But the dragon does not want his toy to make a mess. So he sits down and tries to fix it. 
  He is not happy with the toy anymore. From the bottom of his heart, the red dragon feels sad. His toy is sad too. Even though the big dragon thinks he is a


[Prompt Input]: Write a fantasy story for a 7 year old about a little dragon who wants to be a hero.
--- Hasil Generasi (Top-p / Nucleus Sampling) ---
Write a fantasy story for a 7 year old about a little dragon who wants to be a 

In [ ]:
try:
    test_args = TrainingArguments(
        output_dir="./gpt2-storybook",
        overwrite_output_dir=True,
        num_train_epochs=1,
        per_device_train_batch_size=4,
        save_steps=500,
        save_total_limit=2,
        logging_steps=100,
    )
    print("TrainingArguments initialized successfully.")
except Exception as e:
    import traceback
    print("Error in TrainingArguments:")
    traceback.print_exc()

Error in TrainingArguments:


Traceback (most recent call last):
  File "/tmp/ipykernel_6307/139846363.py", line 2, in <cell line: 0>
    test_args = TrainingArguments(
                ^^^^^^^^^^^^^^^^^
NameError: name 'TrainingArguments' is not defined
